# Research notebook template

Copy this file to start a new notebook (`cp _template.ipynb my_analysis.ipynb`).

It demonstrates the storage-encapsulation patterns codified in REQ_123 / REQ_124 / REQ_125:

- No file-path literals (`Path("results/...")`, `Path("data/...")`)
- No direct `ArtifactLoader(...)` instantiation — reach a configured loader via `variant.artifacts`
- Access stored data through the API: `variant.summary`, `family.variant_registry`, `variant.artifacts.load_*`

See `docs/PROJECT.md` invariant #3 for the rationale.


## Imports

In [ ]:
import numpy as np
import plotly.graph_objects as go

from miscope import load_family


## Variants under study

Curated list of `(prime, seed, data_seed, label)` tuples. The label is used in
figure legends / titles; the parameter tuple identifies the trained variant. This
pattern matches `parameter_space_pca.ipynb` and `parameter_trajectory_pca.ipynb`.


In [ ]:
FAMILY_NAME = "modulo_addition_1layer"

# (prime, model_seed, data_seed, label)
STUDY = [
    (113, 999, 598, "p113 canon"),
    (109, 485, 598, "p109 reference healthy"),
]

family = load_family(FAMILY_NAME)
variants = [
    family.get_variant(prime=p, seed=s, data_seed=ds)
    for (p, s, ds, _label) in STUDY
]
labels = [label for (*_, label) in STUDY]

for v, label in zip(variants, labels):
    print(f"{label:30s}  {v.name:30s}  state={v.state.value}")


## Artifact access

Three load shapes via `variant.artifacts`:

| Method | When to use |
| --- | --- |
| `load_epoch(name, epoch)` | A single epoch of a per-epoch analyzer |
| `load_summary(name)` | A summary artifact (e.g. `repr_geometry`) |
| `load_cross_epoch(name)` | A cross-epoch analyzer's full output |

Use `loader.get_available_analyzers()` / `loader.get_epochs(name)` to introspect.


In [ ]:
variant = variants[0]
loader = variant.artifacts  # configured loader for this variant

available = loader.get_available_analyzers()
print(f"Available analyzers ({len(available)}): {available[:6]}{'...' if len(available) > 6 else ''}")

# Per-epoch — pick the last available epoch for a per-epoch analyzer
epochs = sorted(loader.get_epochs("parameter_snapshot"))
snapshot = loader.load_epoch("parameter_snapshot", epochs[-1])
print(f"parameter_snapshot @ epoch {epochs[-1]}: keys={list(snapshot.keys())[:5]}...")

# Summary — single artifact aggregating across epochs (when the analyzer publishes one)
repr_geom = loader.load_summary("repr_geometry")
print(f"repr_geometry summary: keys={list(repr_geom.keys())[:5]}...")

# Cross-epoch — analyzer that consumes per-epoch artifacts and emits trajectory-level results
ndyn = loader.load_cross_epoch("neuron_dynamics")
print(f"neuron_dynamics: n_epochs={len(ndyn['epochs'])}, d_mlp={ndyn['dominant_freq'].shape[1]}")


## Variant summary and family registry

`variant.summary` returns the parsed `variant_summary.json` for a single variant
(grokking timing, learned frequencies, classification, etc.). `family.variant_registry`
returns the compiled registry — one entry per analyzed variant in the family. Both read
on access; assign to a variable if you'll use them repeatedly.


In [ ]:
summary = variant.summary
print(f"{variant.name}: classification={summary.get('performance_classification', ['?'])[0]}, "
      f"second_descent_onset_epoch={summary.get('second_descent_onset_epoch')}")

registry = family.variant_registry
print(f"\nFamily registry: {len(registry)} entries")
for entry in registry[:3]:
    print(f"  p{entry['prime']:>3} s{entry['model_seed']} ds{entry['data_seed']} -> {entry.get('failure_mode', '?')}")


## Plotting

Minimal Plotly anchor — test-loss curves overlaid per variant. Real analyses replace
this with view-catalog renderers or custom figures, but the loading shape stays the
same: pull data via `variant.metadata` / `variant.artifacts` / `variant.summary`,
then build the figure.


In [ ]:
fig = go.Figure()
for v, label in zip(variants, labels):
    test = np.asarray(v.test_losses)
    fig.add_trace(go.Scatter(
        x=np.arange(len(test)),
        y=test,
        mode="lines",
        name=label,
    ))

fig.update_layout(
    title="Test loss — variants under study",
    xaxis_title="epoch",
    yaxis_title="test loss",
    yaxis_type="log",
    height=400,
)
fig.show()


## Why this shape

The on-disk layout (where checkpoints, artifacts, summaries, and registries live)
is internal to the API. Consumers — including this notebook — reach data only
through `Variant`, `ModelFamily`, and `variant.artifacts`. When the storage layer
moves (e.g. to a remote backend, a different cache, or a new schema), this
notebook keeps working without edits.

If you need a piece of stored data that doesn't have an accessor yet, the right
move is to add the accessor to the API rather than reach past it. See `docs/PROJECT.md`
invariant #3.
